In [6]:
import json
import os
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
 
import pandas as pd
import requests
from dotenv import load_dotenv
 
BASE_URL = "https://api.swedavia.se/flightinfo/v2"
RAW_DIR = Path("data/raw")
 
# Swedavia airports (section 1.3.1 in the docs)
AIRPORTS = ["ARN", "BMA", "GOT", "MMX", "LLA", "UME", "OSD", "VBY", "RNB", "KRN"]
FLIGHT_TYPES = ("arrivals", "departures")
 
load_dotenv()
API_KEY = os.getenv("SWEDAVIA_API_KEY")
if API_KEY is None:
    raise RuntimeError("SWEDAVIA_API_KEY not found, check your .env")
 
HEADERS = {
    "Accept": "application/json",
    "Ocp-Apim-Subscription-Key": API_KEY,
}
 
 
def get_flights(airport: str, flight_type: str, flight_date: date) -> dict:
    """Fetch arrivals or departures for one airport on one date (UTC)."""
    if flight_type not in FLIGHT_TYPES:
        raise ValueError(f"flight_type must be one of {FLIGHT_TYPES}, got {flight_type!r}")
 
    url = f"{BASE_URL}/{airport}/{flight_type}/{flight_date.isoformat()}"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()
 
 
def save_raw(data: dict, flight_type: str, airport: str, flight_date: date) -> Path:
    """Save the untouched API response as JSON (bronze layer)."""
    path = RAW_DIR / flight_type / airport / f"{flight_date.isoformat()}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return path
 
 
def flights_to_df(data: dict, airport: str, flight_type: str) -> pd.DataFrame:
    """Flatten the nested flights list into a DataFrame."""
    # Docs use PascalCase, the JSON is usually camelCase, so handle both
    flights = data.get("flights") or data.get("Flights") or []
    df = pd.json_normalize(flights)
    df["airport"] = airport
    df["flight_type"] = flight_type
    return df
 
 
def fetch_all(flight_type: str, flight_date: date, airports: list[str] = AIRPORTS) -> pd.DataFrame:
    """Fetch, save and combine one flight type for all airports on one date."""
    frames = []
    for airport in airports:
        try:
            data = get_flights(airport, flight_type, flight_date)
        except requests.HTTPError as e:
            print(f"{flight_type} {airport}: failed ({e})")
            continue
 
        save_raw(data, flight_type, airport, flight_date)
        df = flights_to_df(data, airport, flight_type)
        print(f"{flight_type} {airport}: {len(df)} flights")
        frames.append(df)
 
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)
 
 
def yesterday_utc() -> date:
    """The API expects dates in UTC, so don't use date.today()."""
    return datetime.now(timezone.utc).date() - timedelta(days=1)
 
 
def two_days_ago_utc() -> date:
    #TODO: FIX so this so it gets 3 days instead of two
    """The API expects dates in UTC, so don't use date.today()."""
    return datetime.now(timezone.utc).date() - timedelta(days=2)

flight_date = yesterday_utc()
results = {flight_type: fetch_all(flight_type, flight_date) for flight_type in FLIGHT_TYPES}

arrivals ARN: 353 flights
arrivals BMA: 22 flights
arrivals GOT: 82 flights
arrivals MMX: 23 flights
arrivals LLA: 15 flights
arrivals UME: 14 flights
arrivals OSD: 6 flights
arrivals VBY: 13 flights
arrivals RNB: 6 flights
arrivals KRN: 2 flights
departures ARN: 354 flights
departures BMA: 22 flights
departures GOT: 79 flights
departures MMX: 23 flights
departures LLA: 14 flights
departures UME: 14 flights
departures OSD: 6 flights
departures VBY: 13 flights
departures RNB: 6 flights
departures KRN: 2 flights


In [8]:
for flight_type, df in results.items():
    print(flight_type, df.shape)

df

arrivals (536, 34)
departures (533, 40)


,flightId,arrivalAirportSwedish,arrivalAirportEnglish,codeShareData,viaDestinations,remarksEnglish,remarksSwedish,diIndicator,airlineOperator.iata,airlineOperator.icao,...,locationAndStatus.gateOpenUtc,locationAndStatus.gateCloseUtc,checkIn.checkInStatus,checkIn.checkInStatusSwedish,checkIn.checkInStatusEnglish,flightLegIdentifier.aircraftRegistration,flightLegIdentifier.ssrCode,departureTime.estimatedUtc,airport,flight_type
0,FR970,Zadar,Zadar,[],[],[],[],S,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,departures
1,FR4465,Birmingham,Birmingham,[],[],[],[],I,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,departures
2,FR3009,Dublin,Dublin,[],[],[],[],I,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,departures
3,FR4116,Gdansk,Gdansk,[],[],[],[],S,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,departures
4,FR9448,Manchester,Manchester,[],[],[],[],I,FR,RYR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARN,departures
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,SK1130,Stockholm ARN,Stockholm ARN,"[BT6210, KL8541]",[],"[{'text': 'Departed 16:06', 'indicator': 'NEUT...","[{'text': 'Startat 16:06', 'indicator': 'NEUTR...",D,SK,SAS,...,2026-10-08T13:26:47Z,2026-10-08T13:46:22Z,C,Stängd,Closed,SEMKK,NaN,NaN,RNB,departures
529,SK1138,Stockholm ARN,Stockholm ARN,[KL8547],[],"[{'text': 'Departed 18:34', 'indicator': 'NEUT...","[{'text': 'Startat 18:34', 'indicator': 'NEUTR...",D,SK,SAS,...,2026-10-08T16:01:28Z,2026-10-08T16:19:50Z,C,Stängd,Closed,SEMKC,NaN,NaN,RNB,departures
530,SK1128,Stockholm ARN,Stockholm ARN,[],[],"[{'text': 'Departed 20:37', 'indicator': 'NEUT...","[{'text': 'Startat 20:37', 'indicator': 'NEUTR...",D,SK,SAS,...,2026-10-08T17:46:41Z,2026-10-08T18:16:57Z,C,Stängd,Closed,SEMKE,NaN,NaN,RNB,departures
531,SK1045,Stockholm ARN,Stockholm ARN,"[AF4892, BT6195, EY3989, KL8485, LO5438]",[],"[{'text': 'Departed 13:57', 'indicator': 'NEUT...","[{'text': 'Startat 13:57', 'indicator': 'NEUTR...",D,SK,SAS,...,NaN,2026-10-08T11:53:46Z,C,Stängd,Closed,SEROM,NaN,NaN,KRN,departures


In [9]:
import json
import os
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
 
import pandas as pd
import requests
from dotenv import load_dotenv
 
BASE_URL = "https://api.swedavia.se/flightinfo/v2"
RAW_DIR = Path("data/raw")
 
# Swedavia airports (section 1.3.1 in the docs)
AIRPORTS = ["ARN", "BMA", "GOT", "MMX", "LLA", "UME", "OSD", "VBY", "RNB", "KRN"]
FLIGHT_TYPES = ("arrivals", "departures")
 
load_dotenv()
API_KEY = os.getenv("SWEDAVIA_API_KEY")
if API_KEY is None:
    raise RuntimeError("SWEDAVIA_API_KEY not found, check your .env")
 
HEADERS = {
    "Accept": "application/json",
    "Ocp-Apim-Subscription-Key": API_KEY,
}
 
 
def get_flights(airport: str, flight_type: str, flight_date: date) -> dict:
    """Fetch arrivals or departures for one airport on one date (UTC)."""
    if flight_type not in FLIGHT_TYPES:
        raise ValueError(f"flight_type must be one of {FLIGHT_TYPES}, got {flight_type!r}")
 
    url = f"{BASE_URL}/{airport}/{flight_type}/{flight_date.isoformat()}"
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()
 
 
def save_raw(data: dict, flight_type: str, airport: str, flight_date: date) -> Path:
    """Save the untouched API response as JSON (bronze layer)."""
    path = RAW_DIR / flight_type / airport / f"{flight_date.isoformat()}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return path
 
 
def flights_to_df(data: dict, airport: str, flight_type: str) -> pd.DataFrame:
    """Flatten the nested flights list into a DataFrame."""
    # Docs use PascalCase, the JSON is usually camelCase, so handle both
    flights = data.get("flights") or data.get("Flights") or []
    df = pd.json_normalize(flights)
    df["airport"] = airport
    df["flight_type"] = flight_type
    return df
 
 
def fetch_all(flight_type: str, flight_date: date, airports: list[str] = AIRPORTS) -> pd.DataFrame:
    """Fetch, save and combine one flight type for all airports on one date."""
    frames = []
    for airport in airports:
        try:
            data = get_flights(airport, flight_type, flight_date)
        except requests.HTTPError as e:
            # Print the response body too, the API often explains what went wrong
            print(f"{flight_date} {flight_type} {airport}: failed ({e})")
            print(f"  response: {e.response.text[:300]}")
            continue
 
        save_raw(data, flight_type, airport, flight_date)
        df = flights_to_df(data, airport, flight_type)
        df["flight_date"] = flight_date
        print(f"{flight_date} {flight_type} {airport}: {len(df)} flights")
        frames.append(df)
 
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)
 
 
def days_back_utc(n_days: int = 3) -> list[date]:
    """Return the last n_days in UTC, oldest first (e.g. 3, 2, 1 days ago)."""
    today = datetime.now(timezone.utc).date()
    return [today - timedelta(days=d) for d in range(n_days, 0, -1)]
 
 
def fetch_period(n_days: int = 3) -> dict[str, pd.DataFrame]:
    """Fetch all flight types for the last n_days, one DataFrame per flight type."""
    frames = {flight_type: [] for flight_type in FLIGHT_TYPES}
 
    for flight_date in days_back_utc(n_days):
        print(f"\n=== Fetching {flight_date} ===")
        for flight_type in FLIGHT_TYPES:
            frames[flight_type].append(fetch_all(flight_type, flight_date))
 
    return {ft: pd.concat(dfs, ignore_index=True) for ft, dfs in frames.items()}

In [12]:
results = fetch_period(n_days=2)
 
for flight_type, df in results.items():
    print(flight_type, df.shape)


=== Fetching 2026-10-07 ===
2026-10-07 arrivals ARN: 349 flights
2026-10-07 arrivals BMA: 23 flights
2026-10-07 arrivals GOT: 77 flights
2026-10-07 arrivals MMX: 22 flights
2026-10-07 arrivals LLA: 15 flights
2026-10-07 arrivals UME: 15 flights
2026-10-07 arrivals OSD: 6 flights
2026-10-07 arrivals VBY: 12 flights
2026-10-07 arrivals RNB: 6 flights
2026-10-07 arrivals KRN: 2 flights
2026-10-07 departures ARN: 346 flights
2026-10-07 departures BMA: 23 flights
2026-10-07 departures GOT: 76 flights
2026-10-07 departures MMX: 22 flights
2026-10-07 departures LLA: 15 flights
2026-10-07 departures UME: 16 flights
2026-10-07 departures OSD: 7 flights
2026-10-07 departures VBY: 12 flights
2026-10-07 departures RNB: 6 flights
2026-10-07 departures KRN: 2 flights

=== Fetching 2026-10-08 ===
2026-10-08 arrivals ARN: 353 flights
2026-10-08 arrivals BMA: 22 flights
2026-10-08 arrivals GOT: 82 flights
2026-10-08 arrivals MMX: 23 flights
2026-10-08 arrivals LLA: 15 flights
2026-10-08 arrivals UME: 